# Merge individual CSV files per website into a single CSV per website

In [ ]:
import pandas as pd
import glob
import os

# Path to the folder containing CSV files
ruta_carpeta = "data-website-searches/links/links_stackoverflow"  # Adjust for each website

# Output path inside the project
ruta_salida = os.path.join("data-website-searches/links", "links_stackoverflow.csv")  # Adjust for each website

# Find all CSV files in the folder
archivos_csv = glob.glob(os.path.join(ruta_carpeta, "*.csv"))

# List to store DataFrames
dataframes = []

for archivo in archivos_csv:
    df = pd.read_csv(archivo)
    dataframes.append(df)

# Merge all DataFrames
df_final = pd.concat(dataframes, ignore_index=True)

# Remove duplicates if necessary
df_final.drop_duplicates(inplace=True)

# Save to output path
df_final.to_csv(ruta_salida, index=False)

print(f"Merged CSV saved at: {ruta_salida}")


# Add "website" column to each CSV per website

In [ ]:
import pandas as pd
import os

# Script configuration

# Name of the original file (must be inside 'data-website-searches/links')
nombre_archivo_original = "links_stackoverflow.csv"  # Adjust for each CSV

# Value to assign to the new 'website' column
valor_website = "stackoverflow"  # Adjust for each website

# Paths

# Input file path
ruta_entrada = os.path.join("data-website-searches/links", nombre_archivo_original)

# Load CSV
df = pd.read_csv(ruta_entrada)

# Add new column
df["website"] = valor_website

# Create output file name
nombre_base = os.path.splitext(nombre_archivo_original)[0]
nuevo_nombre_archivo = f"{nombre_base}_withwebsitecolumn.csv"

# Output folder
carpeta_salida = "data-website-searches/linkswithwebsitecolumn"
os.makedirs(carpeta_salida, exist_ok=True)

# Full output path
ruta_salida = os.path.join(carpeta_salida, nuevo_nombre_archivo)

# Save new CSV
df.to_csv(ruta_salida, index=False)

print(f"New CSV saved at: {ruta_salida}")


# Merge all CSV files with "website" column into a single unified CSV

In [ ]:
import pandas as pd
import os

# Input folder
carpeta_entrada = "data-website-searches/linkswithwebsitecolumn"

# Output folder
carpeta_salida = "data-website-searches/linksall"
os.makedirs(carpeta_salida, exist_ok=True)

# List to store DataFrames
dfs = []

# Iterate through all CSV files
for archivo in os.listdir(carpeta_entrada):
    if archivo.endswith(".csv"):
        ruta = os.path.join(carpeta_entrada, archivo)
        df = pd.read_csv(ruta)
        dfs.append(df)

# Merge all DataFrames
df_total = pd.concat(dfs, ignore_index=True)

# Save output
archivo_salida = os.path.join(carpeta_salida, "links_all.csv")
df_total.to_csv(archivo_salida, index=False)

print(f"Merged file saved at: {archivo_salida}")


# Add "httpcode" column and retrieve HTTP status for each link

In [ ]:
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import os

# Paths
archivo_entrada = "data-website-searches/linksall/links_all.csv"
carpeta_salida = "data-website-searches/linksallwithhttpcode"
archivo_salida = os.path.join(carpeta_salida, "links_all_withhttpcode.csv")

# Create output folder if needed
os.makedirs(carpeta_salida, exist_ok=True)

# Load dataset
df = pd.read_csv(archivo_entrada)

# Function to get HTTP status code
def obtener_http_code(url):
    try:
        response = requests.head(url, allow_redirects=True, timeout=5, headers={"User-Agent": "Mozilla/5.0"})
        return response.status_code
    except requests.RequestException:
        return None

# URL list
urls = df["Link"].tolist()

# Results list
http_codes = [None] * len(urls)

# Parallel execution
max_workers = 20
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_index = {executor.submit(obtener_http_code, url): i for i, url in enumerate(urls)}

    for future in tqdm(as_completed(future_to_index), total=len(urls), desc="Fetching HTTP codes"):
        i = future_to_index[future]
        try:
            http_codes[i] = future.result()
        except Exception:
            http_codes[i] = None

# Add column
df["httpcode"] = http_codes

# Save output
df.to_csv(archivo_salida, index=False)

print(f"Process completed. File saved at: {archivo_salida}")


# Filter unique links and keep only HTTP 200 responses

In [ ]:
import pandas as pd
import os

# Input and output paths
archivo_entrada = "data-website-searches/linksallwithhttpcode/links_all_withhttpcode.csv"
carpeta_salida = "data-website-searches/linksallwithhttpcodeonlyunique200"
archivo_salida = os.path.join(carpeta_salida, "links_all_withhttpcode_onlyunique200.csv")

# Create output folder if needed
os.makedirs(carpeta_salida, exist_ok=True)

# Load CSV
df = pd.read_csv(archivo_entrada)

# 1) Remove duplicates in "Link" column
df = df.drop_duplicates(subset="Link", keep="first")

# 2) Keep only httpcode == 200 or 200.0
df = df[df["httpcode"].isin([200, 200.0])]

# Save filtered CSV
df.to_csv(archivo_salida, index=False)

print(f"Filtered file saved at: {archivo_salida}")
print(f"Total links after filtering: {len(df)}")
